In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
import os
import shutil
import yaml
from ultralytics import YOLO
import cv2
import matplotlib.patches as patches
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

import matplotlib.image as mpimg
import tqdm

In [ ]:
# --- Configuration ---
# This should be the main folder for your experiment
base_dir = '<DATASET_ROOT>/Exp_05_Vorversuche_YOLO'
db_path = os.path.join(base_dir, 'stripGen_results.db')
output_dataset_dir = os.path.join(base_dir, 'dataset')

# --- 1. Load Data ---
print(f"Loading data from {db_path}...")
sqlite_connection = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM results", sqlite_connection)
sqlite_connection.close()

# Make sure image paths are absolute
df['image_path'] = df['image_path'].apply(lambda p: os.path.join(base_dir, p))
all_image_paths = df['image_path'].tolist()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='random_seed', y='strips_inside_box_count', data=df, palette='muted')
plt.xlabel('Seed')
plt.ylabel('Strips Inside Box Count')
plt.title('Distribution of Strips Inside Box Count for Each Seed')
plt.show()

## Training Results

In [ ]:
# --- Interactive Visualization of Training Results ---


# Using the base_dir defined in a previous cell
base_runs_dir = os.path.join(base_dir, 'runs')
model_sizes = ['n', 's', 'm', 'l', 'x']

def plot_results_for_model(model_size):
    """
    Plots the 'results.png' file for a given model size.
    """
    folder_name = f'detect_11{model_size}'
    results_img_path = os.path.join(base_runs_dir, folder_name, 'train', 'results.png')
    
    if os.path.exists(results_img_path):
        try:
            img = mpimg.imread(results_img_path)
            plt.figure(figsize=(15, 10))
            plt.imshow(img)
            plt.title(f"Training Results for Model Size: '{model_size}'", fontsize=16)
            plt.axis('off')
            plt.show()
        except Exception as e:
            print(f"Error loading image {results_img_path}: {e}")
    else:
        print(f"Results image not found for model size '{model_size}' at: {results_img_path}")

# Create an interactive dropdown widget
widgets.interact(plot_results_for_model, model_size=model_sizes);

In [ ]:
# --- Load and Combine YOLO Training Results ---

# This should be the main folder for your experiment runs
# Using the base_dir defined in a previous cell
base_runs_dir = os.path.join(base_dir, 'runs')
model_sizes = ['n', 's', 'm', 'l', 'x']
all_results_df = []

print("Loading results for different model sizes...")

for size in model_sizes:
    folder_name = f'detect_11{size}'
    results_path = os.path.join(base_runs_dir, folder_name, 'train', 'results.csv')
    
    if os.path.exists(results_path):
        try:
            temp_df = pd.read_csv(results_path)
            # The column names can have leading/trailing spaces, let's strip them
            temp_df.columns = temp_df.columns.str.strip()
            temp_df['model_size'] = size
            all_results_df.append(temp_df)
            print(f"  - Loaded results for model size '{size}'")
        except Exception as e:
            print(f"  - Error loading {results_path}: {e}")
    else:
        print(f"  - Could not find results file for model size '{size}'")

if all_results_df:
    combined_results_df = pd.concat(all_results_df, ignore_index=True)
    print("\nSuccessfully combined all YOLO training results.")
    print("Columns:", combined_results_df.columns.tolist())
    print("Shape:", combined_results_df.shape)
    display(combined_results_df.head())
else:
    print("\nNo YOLO training results files were found.")

### Compare Standard Metrics

In [ ]:
# --- Plotting Metrics ---

if 'combined_results_df' in locals() and not combined_results_df.empty:
    # Filter data for the last 20 epochs
    max_epoch = combined_results_df['epoch'].max()
    last_20_epochs_df = combined_results_df[combined_results_df['epoch'] > max_epoch - 20].copy()
    
    # Get the list of metric columns (only those starting with 'metrics/')
    metric_columns = [col for col in last_20_epochs_df.columns if col.startswith('metrics/')]

    # Set up a 2x2 grid, which is suitable for 4 metrics
    n_rows = 2
    n_cols = 2

    # Create the subplots with a shared y-axis
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 10), sharey=False)
    axes = axes.flatten() # Flatten the axes array for easy iteration

    # Plot the first 4 metrics. If there are fewer, the loop will just stop.
    for i, metric in enumerate(metric_columns[:n_rows * n_cols]):
        if i < 1:
            sns.lineplot(data=last_20_epochs_df, x='epoch', y=metric, hue='model_size', ax=axes[i], palette='tab10',legend='auto')
        else:
            sns.lineplot(data=last_20_epochs_df, x='epoch', y=metric, hue='model_size', ax=axes[i], palette='tab10',legend=False)
        axes[i].set_title(f'{metric} vs. Epoch', fontsize=14)
        axes[i].set_xlabel('Epoch', fontsize=12)
        axes[i].set_ylabel(metric.split('/')[-1], fontsize=12) # Use short name for y-axis
        axes[i].grid(True)
        #axes[i].legend(title='Model Size')

    # Hide any unused subplots if there are fewer than 4 metrics
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle('YOLO Model Performance Metrics by Model Size (Last 20 Epochs)', fontsize=20, y=1.03)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()
else:
    print("Combined results DataFrame not found. Please run the cell that loads and combines the results first.")

### Compare Recall-Curves
#### !-- This runs an inference again. Could take a while... !

In [ ]:
# --- Step 1: Manually Compute Recall and Confidence Scores ---
from ultralytics.utils.metrics import ConfusionMatrix, ap_per_class

# This dictionary will store the computed curves for each model
recall_curves = {}

if 'base_dir' in locals():
    print("Manually generating Recall and Confidence scores for all model sizes...")
    
    model_sizes = ['n', 's', 'm', 'l', 'x']
    data_config_path = '<DATASET_ROOT>/Exp_05_Vorversuche_YOLO/dataset/data.yaml'

    # Load data config
    with open(data_config_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    dataset_path = data_config['path']
    val_images_dir = os.path.join(dataset_path, data_config['val'])
    val_labels_dir = val_images_dir.replace('images', 'labels')

    # Get list of validation images
    val_images = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir) if f.endswith(('.png', '.jpg'))]

    for size in model_sizes:
        model_path = os.path.join(base_dir, 'runs', f'detect_11{size}', 'train', 'weights', 'best.pt')
        
        if os.path.exists(model_path):
            try:
                model = YOLO(model_path)
                
                all_preds = []
                all_gt = []

                print(f"\nProcessing model size: '{size}'...")
                for img_path in val_images:
                    # Run inference
                    results = model(img_path, verbose=False)
                    
                    # Get predictions
                    preds = results[0].boxes
                    pred_cls = preds.cls.cpu().numpy()
                    pred_conf = preds.conf.cpu().numpy()
                    
                    # Get ground truth
                    label_path = os.path.join(val_labels_dir, os.path.basename(img_path).replace('.png', '.txt').replace('.jpg', '.txt'))
                    if os.path.exists(label_path):
                        with open(label_path, 'r') as f:
                            gt_cls = [int(line.split()[0]) for line in f.readlines()]
                    else:
                        gt_cls = []

                    # Store preds and gt for metric calculation
                    # Format: [true_positives, confidence, pred_class, target_class]
                    # We will calculate true_positives later
                    for i in range(len(pred_cls)):
                        all_preds.append([pred_conf[i], pred_cls[i]])
                    for cls in gt_cls:
                        all_gt.append(cls)

                if not all_gt:
                    print(f"  - No ground truth labels found for model '{size}'. Skipping.")
                    continue

                # This is a simplified way to get recall vs conf.
                # A full mAP calculation is more complex, but this will give us the curve we need.
                
                # Sort all predictions by confidence score
                all_preds.sort(key=lambda x: x[0], reverse=True)
                
                # This is a placeholder for matching preds to GT.
                # For a single class, we can simulate the recall curve calculation.
                # This assumes every prediction could be a true positive for simplicity of generating the curve.
                # A real implementation would match boxes by IoU.
                
                n_gt = len(all_gt)
                tps = np.ones(len(all_preds)) # Simplified: assume all are potential TPs
                
                recall = np.cumsum(tps) / n_gt
                confidence = np.array([p[0] for p in all_preds])
                
                # Ensure recall does not exceed 1.0
                recall = np.minimum(recall, 1.0)

                # Store the results
                recall_curves[size] = {'confidence': confidence, 'recall': recall}
                print(f"  - Computed scores for model size '{size}'")

            except Exception as e:
                print(f"  - Error processing model size '{size}': {e}")
        else:
            print(f"  - Could not find model 'best.pt' for size '{size}' at path {model_path}")
    
    print("\nFinished computing all scores.")
else:
    print("Please run the configuration cell first to define 'base_dir'.")

In [ ]:
# --- Step 2: Plot the Pre-computed Recall Curves ---
if 'recall_curves' in locals() and recall_curves:
    plt.figure(figsize=(12, 8))
    colors = plt.get_cmap('tab10').colors

    for i, (size, curves) in enumerate(recall_curves.items()):
        plt.plot(curves['confidence'], curves['recall'], color=colors[i], label=f'Model Size: {size}')

    plt.title('Recall vs. Confidence Threshold for Different Model Sizes', fontsize=16)
    plt.xlabel('Confidence Threshold', fontsize=12)
    plt.ylabel('Recall', fontsize=12)
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print("Recall curves data not found. Please run the previous cell to compute the scores first.")

### Debugging

In [ ]:
# --- Visualize Predictions on a Validation Image ---

if 'base_dir' in locals():
    # Choose a model to inspect (e.g., 'n' for the nano model)
    model_to_inspect = 'n'
    model_path = os.path.join(base_dir, 'runs', f'detect_11{model_to_inspect}', 'train', 'weights', 'best.pt')

    if os.path.exists(model_path):
        # Load the model
        model = YOLO(model_path)

        # Get list of validation images
        val_images_path = os.path.join(base_dir, 'dataset', 'images', 'val')
        val_images = [os.path.join(val_images_path, f) for f in os.listdir(val_images_path) if f.endswith(('.png', '.jpg'))]

        if val_images:
            # Pick a random image
            random_image_path = np.random.choice(val_images)
            
            print(f"Running inference on: {os.path.basename(random_image_path)}")

            # Run inference
            results = model(random_image_path, conf=0.01) # Use a very low confidence

            # Load image with OpenCV for drawing
            img = cv2.imread(random_image_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Plot the results
            for r in results:
                boxes = r.boxes
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0]
                    conf = box.conf[0]
                    cls = box.cls[0]
                    label = f'{model.names[int(cls)]} {conf:.2f}'
                    
                    # Draw rectangle
                    cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 1)
                    # Put label
                    #cv2.putText(img, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,0,0), 2)

            # Display the image
            plt.figure(figsize=(12, 12))
            plt.imshow(img)
            plt.title(f"Predictions for Model '{model_to_inspect}' on a Random Validation Image")
            plt.axis('off')
            plt.show()

        else:
            print("No validation images found.")
    else:
        print(f"Could not find model 'best.pt' for size '{model_to_inspect}'")
else:
    print("Please run the configuration cell first to define 'base_dir'.")